In [1]:
import os
import cv2
import xml.etree.ElementTree as ET
import numpy as np
from ultralytics import YOLO

def parse_voc_xml(xml_path):
    """Parse Pascal VOC XML annotation file and return bounding boxes in (x1, y1, x2, y2) format."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    boxes = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        boxes.append([xmin, ymin, xmax, ymax])
    
    return boxes

def iou(box1, box2):
    """Compute Intersection over Union (IoU) between two bounding boxes."""
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1 = max(x1, x1g)
    yi1 = max(y1, y1g)
    xi2 = min(x2, x2g)
    yi2 = min(y2, y2g)

    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)

    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def evaluate_model(model_path, images_folder, annotations_folder):
    """Evaluate YOLOv8 model on a test set using IoU."""
    model = YOLO(model_path)
    image_files = [f for f in os.listdir(images_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

    iou_scores = []

    for img_file in image_files:
        img_path = os.path.join(images_folder, img_file)
        annotation_path = os.path.join(annotations_folder, img_file.replace('.png', '.xml').replace('.jpg', '.xml'))

        if not os.path.exists(annotation_path):
            continue

        # Load ground truth
        gt_bboxes = parse_voc_xml(annotation_path)

        # Run YOLOv8 model
        results = model(img_path)
        pred_bboxes = [[int(box[0]), int(box[1]), int(box[2]), int(box[3])] for box in results[0].boxes.xyxy]

        # Compute IoU for each prediction with ground truth
        for pred in pred_bboxes:
            best_iou = 0
            for gt in gt_bboxes:
                best_iou = max(best_iou, iou(pred, gt))
            if best_iou > 0:
                iou_scores.append(best_iou)

    mean_iou = np.mean(iou_scores) if iou_scores else 0
    print(f"Mean IoU: {mean_iou:.4f}")

# Example usage
evaluate_model("best.pt", "images", "annotations")


FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'